In [4]:
# ============================================================
# CHENNAI EV CHARGING STATION - ML MODEL
# Random Forest Classification
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. LOAD CANDIDATE DATA
# ------------------------------------------------------------

df = pd.read_csv(r"C:\Users\krish\Downloads\Chennai_EV_GIS_Project\QGIS\New folder\EV_Candiate_Sampled.csv")

print("Dataset shape:", df.shape)
print(df.head())


# ------------------------------------------------------------
# 2. FEATURES
# ------------------------------------------------------------

features = [
    "POP_1",
    "ROAD_1",
    "POPS_1",
    "POPVAL_1"
]

X = df[features].copy()


# ------------------------------------------------------------
# 3. TARGET
# ------------------------------------------------------------

# IMPORTANT:
# This version uses SUIT_CLASS only as a demonstration.
# For a scientifically valid ML model, use an independent
# target such as existing-station presence.

df["TARGET"] = df["SUIT_CLASS"].map({
    "Medium": 0,
    "High": 1
})

y = df["TARGET"]


# ------------------------------------------------------------
# 4. REMOVE MISSING VALUES
# ------------------------------------------------------------

data = pd.concat([X, y], axis=1).dropna()

X = data[features]
y = data["TARGET"]


# ------------------------------------------------------------
# 5. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ------------------------------------------------------------
# 6. RANDOM FOREST MODEL
# ------------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)


# ------------------------------------------------------------
# 7. PREDICTION
# ------------------------------------------------------------

y_pred = model.predict(X_test)


# ------------------------------------------------------------
# 8. MODEL EVALUATION
# ------------------------------------------------------------

accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


# ------------------------------------------------------------
# 9. PREDICT ALL CANDIDATE LOCATIONS
# ------------------------------------------------------------

df["ML_PREDICTION"] = model.predict(
    df[features]
)

df["ML_PROBABILITY"] = model.predict_proba(
    df[features]
)[:, 1]


# ------------------------------------------------------------
# 10. CONVERT TO CLASS
# ------------------------------------------------------------

df["ML_CLASS"] = df["ML_PREDICTION"].map({
    0: "Medium",
    1: "High"
})


# ------------------------------------------------------------
# 11. SORT BY ML PROBABILITY
# ------------------------------------------------------------

df = df.sort_values(
    "ML_PROBABILITY",
    ascending=False
)


# ------------------------------------------------------------
# 12. SAVE RESULT
# ------------------------------------------------------------

df.to_csv(
    r"C:\Users\krish\Downloads\Chennai_EV_GIS_Project\ML_Output\Chennai_EV_ML_Predictions.csv",
    index=False
)

print("\nML prediction file created:")
print("Chennai_EV_ML_Predictions.csv")

print("\nTop candidate locations:")
print(
    df[
        [
            "rand_point_id",
            "POPVAL_1",
            "ROAD_1",
            "ML_PROBABILITY",
            "ML_CLASS"
        ]
    ].head(20)
)

Dataset shape: (254, 9)
   fid  rand_point_id   DN  POP_1  ROAD_1  POPS_1    POPVAL_1  SUIT_SCORE  \
0   12            0.0  1.0    1.0     5.0     5.0  106.088181         5.0   
1   16            1.0  1.0    1.0     5.0     4.0  102.823647         4.5   
2   26            2.0  1.0    1.0     5.0     5.0  112.290741         5.0   
3   28            3.0  1.0    1.0     5.0     5.0  117.390854         5.0   
4   33            4.0  1.0    1.0     5.0     5.0  107.188622         5.0   

  SUIT_CLASS  
0       High  
1       High  
2       High  
3       High  
4       High  

Model Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00         1
         1.0       1.00      1.00      1.00        50

    accuracy                           1.00        51
   macro avg       1.00      1.00      1.00        51
weighted avg       1.00      1.00      1.00        51


Confusion Matrix:
[[ 1  0]
 [ 0 50]]

ML predictio

In [10]:
# ============================================================
# CHENNAI EV CHARGING LOCATION - ML CLUSTERING
# K-MEANS
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(r"C:\Users\krish\Downloads\Chennai_EV_GIS_Project\QGIS\New folder\EV_Candiate_Sampled.csv")

print("Dataset shape:", df.shape)


# ------------------------------------------------------------
# 2. SELECT USEFUL FEATURES
# ------------------------------------------------------------

features = [
    "POPS_1",
    "POPVAL_1",
    "SUIT_SCORE"
]

X = df[features].copy()

# Remove missing values
valid = X.notna().all(axis=1)

df_ml = df[valid].copy()
X = X[valid]


# ------------------------------------------------------------
# 3. STANDARDIZE FEATURES
# ------------------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


# ------------------------------------------------------------
# 4. K-MEANS CLUSTERING
# ------------------------------------------------------------

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=20
)

df_ml["ML_CLUSTER"] = kmeans.fit_predict(X_scaled)


# ------------------------------------------------------------
# 5. SILHOUETTE SCORE
# ------------------------------------------------------------

silhouette = silhouette_score(
    X_scaled,
    df_ml["ML_CLUSTER"]
)

print("\nSilhouette Score:", round(silhouette, 3))


# ------------------------------------------------------------
# 6. CHECK CLUSTER CHARACTERISTICS
# ------------------------------------------------------------

cluster_summary = df_ml.groupby(
    "ML_CLUSTER"
)[features].mean()

print("\nCluster Summary:")
print(cluster_summary)


# ------------------------------------------------------------
# 7. IDENTIFY HIGH-POTENTIAL CLUSTER
# ------------------------------------------------------------

cluster_summary["mean_score"] = (
    cluster_summary["POPS_1"] +
    cluster_summary["SUIT_SCORE"]
)

high_cluster = cluster_summary[
    "mean_score"
].idxmax()

print("\nHigh-potential cluster:", high_cluster)


# ------------------------------------------------------------
# 8. CREATE ML CLASS
# ------------------------------------------------------------

df_ml["ML_CLASS"] = np.where(
    df_ml["ML_CLUSTER"] == high_cluster,
    "High Potential",
    "Other Potential"
)


# ------------------------------------------------------------
# 9. ML PRIORITY SCORE
# ------------------------------------------------------------

df_ml["ML_PRIORITY_SCORE"] = (
    0.4 * df_ml["POPS_1"] +
    0.3 * df_ml["SUIT_SCORE"] +
    0.3 * (
        df_ml["POPVAL_1"] /
        df_ml["POPVAL_1"].max()
    ) * 5
)


# ------------------------------------------------------------
# 10. RANK LOCATIONS
# ------------------------------------------------------------

df_ml = df_ml.sort_values(
    "ML_PRIORITY_SCORE",
    ascending=False
)

df_ml["ML_RANK"] = range(
    1,
    len(df_ml) + 1
)


# ------------------------------------------------------------
# 11. SAVE FINAL ML OUTPUT
# ------------------------------------------------------------

output_file = r"C:\Users\krish\Downloads\Chennai_EV_GIS_Project\ML_Output\Chennai_EV_ML_Final.csv"

df_ml.to_csv(
    output_file,
    index=False
)

print("\n--------------------------------")
print("ML ANALYSIS COMPLETED")
print("--------------------------------")

print("\nOutput file:")
print(output_file)

print("\nTop 20 potential locations:")

print(
    df_ml[
        [
            "rand_point_id",
            "POPVAL_1",
            "POPS_1",
            "SUIT_SCORE",
            "ML_PRIORITY_SCORE",
            "ML_RANK",
            "ML_CLASS"
        ]
    ].head(20)
)

Dataset shape: (254, 9)

Silhouette Score: 0.746

Cluster Summary:
            POPS_1    POPVAL_1  SUIT_SCORE
ML_CLUSTER                                
0              4.0   97.913697         4.5
1              5.0  114.063623         5.0
2              2.8   63.459689         3.9

High-potential cluster: 1

--------------------------------
ML ANALYSIS COMPLETED
--------------------------------

Output file:
C:\Users\krish\Downloads\Chennai_EV_GIS_Project\ML_Output\Chennai_EV_ML_Final.csv

Top 20 potential locations:
     rand_point_id    POPVAL_1  POPS_1  SUIT_SCORE  ML_PRIORITY_SCORE  \
36            36.0  129.127228     5.0         5.0           5.000000   
174          174.0  129.127228     5.0         5.0           5.000000   
162          162.0  128.829437     5.0         5.0           4.996541   
30            30.0  128.829437     5.0         5.0           4.996541   
29            29.0  128.410660     5.0         5.0           4.991676   
171          171.0  127.826454     5.0 

C:\Users\krish\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
